# MNIST Convolutional Neural Network Example

The MNIST Dataset is a set of images, 28 by 28 pixels. These images contain the numbers 0 - 9 (10 numbers) and the associated label.
We will design a model, such that it is rewarded for mapping image data as input, to a number from 0-9. We will compare the outputs from the model to the known label as part of our error function.

## Imports

In [1]:
import torch
import torchvision
from torchvision import transforms, datasets

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from tqdm import tqdm

## Load Datasets

In [2]:
# Root File Location of the Data:
# TODO: STUDENT TO CHANGE
file_loc = "C:\\Users\\igriffit\\OneDrive - University of South Wales (1)\\Data\\MNIST Data"


# Training Data Location:
load_train = f'{file_loc}\\MNIST_train.pt'
load_train_labels = f'{file_loc}\\MNIST_train_labels.pt'

# Testing Data Location:
load_test = f'{file_loc}\\MNIST_test.pt'
load_test_labels = f'{file_loc}\\MNIST_test_labels.pt'



# Load Training and Testing Data:
# Note: The "to" function, can also be used to change to float.
train_images = torch.load(load_train)
test_images = torch.load(load_test)
train_labels = torch.load(load_train_labels).to(torch.long)
test_labels = torch.load(load_test_labels).to(torch.long)

## View Datasets

In [ ]:
import matplotlib.pyplot as plt

In [3]:
print(train_images.shape)

torch.Size([60000, 1, 28, 28])


In [ ]:
sample_num = 10000
print(f'The Corrosponding Label is: {train_labels[sample_num]}')
plt.imshow(train_images[sample_num][0], cmap='gray')
plt.show()

## Convolutional Neural Network - Step 1

There is some added complexity to creating a convolutional nerual network, where mid-way through the network, we go from processing multi-dimensional data to single dimensional data (tabular data). In order to change the shape of our data, or as we would say, FLATTEN our data, we have to understanding it's shape before we do that.

In [4]:
class Conv_NN(nn.Module):
    def __init__(self, n_channels):
        super().__init__()
        
        # Convolutional Layers & Max Pooling Layer Defined:
        self.conv1 = nn.Conv2d(in_channels = n_channels, out_channels = 8, kernel_size = (5,5) )
        self.conv2 = nn.Conv2d(in_channels = 8, out_channels = 16,  kernel_size = (5,5))
        self.pool = nn.MaxPool2d((2,2))
        
        # Flatten Data: Cannot yet be defined
        
        
        # Linear Layers: Cannot yet be defined
        

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)

        x = F.relu(self.conv2(x))
        x = self.pool(x)

        print(f"Step 1: Shape of Data before Flattening: {x.shape}")


        




        return x


        

In [5]:
net  = Conv_NN(n_channels = 1)

In [6]:
n_samples = 10
test_output = net.forward(train_images[0:n_samples])

Step 1: Shape of Data before Flattening: torch.Size([10, 16, 4, 4])


In the above code, we run a test sample of data thorugh our model before we've trained it, and we can see that the final shape of our data is ```[batch_size, 16, 4, 4]```. Now, to flatten this we can do it in one of two ways:

- Use the following code ```x = x.view(-1, 16*4*4)```: This will flatten our data to be the shape ```[batch_size, n_feature=256]```
- Create a layer that uses the built in flatten 


https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html

In [7]:
# OPTION 1: View Function:
test_output1 = test_output.view(-1, 16*4*4)
print(f"The Shape after using the View Function is: {test_output1.shape}")

# OPTION 2: Flatten Function:

n_dimensions = 1 # What Happens when I change this number?
myfunc = nn.Flatten(n_dimensions)

test_output2 = myfunc(test_output)
print(f"The Shape after using the Flatten Function is: {test_output2.shape}")

The Shape after using the View Function is: torch.Size([10, 256])
The Shape after using the Flatten Function is: torch.Size([10, 256])


## Convolutional Neural Network - Step 2

Now that we understand how to flatten our data, and how many features it is going to flatten into (256) we can finish the structure of the network.

In [11]:
class Conv_NN(nn.Module):
    def __init__(self, n_channels):
        super().__init__()
        
        # Convolutional Layers & Max Pooling Layer Defined:
        self.conv1 = nn.Conv2d(in_channels = n_channels, out_channels = 8, kernel_size = (5,5) )
        self.conv2 = nn.Conv2d(in_channels = 8, out_channels = 16,  kernel_size = (5,5))
        self.pool = nn.MaxPool2d((2,2))
        
        # Flatten Data:
        self.flatten = nn.Flatten(1)
        
        # Linear Layers:
        self.fc1 = nn.Linear(16*4*4, 128)
        self.fc2 = nn.Linear(128, 10)
        
    def forward(self, x):
        # Step 1: Convolutional Layers
        x = F.relu(self.conv1(x))
        x = self.pool(x)

        x = F.relu(self.conv2(x))
        x = self.pool(x)

        # print(f"Step 1: Shape of Data before Flattening: {x.shape}")


        # Step 2: Flatten Data & Linear Layers
        x = self.flatten(x)

        x = F.relu(self.fc1(x))
        x = F.log_softmax(self.fc2(x), dim=1)

        return x

## Loss and Optimizer

In [12]:
net  = Conv_NN(n_channels = 1)

loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.01)

## Training Loop (CPU)

In [13]:
epochs = 5
BATCH_SIZE = 100
for epoch in range (epochs):
    for i in tqdm(range(0,len(train_images), BATCH_SIZE)):
        # Batch Our Data:
        batch_data = train_images[i:i+BATCH_SIZE]
        batch_labels = train_labels[i:i+BATCH_SIZE]
        # Calculate Output:
        net.zero_grad()  
        output = net(batch_data)  
        # CAlculate & Update Gradients:
        loss = loss_function(output, batch_labels) 
        loss.backward()  
        optimizer.step()  
    print(loss)

100%|██████████| 600/600 [00:04<00:00, 145.26it/s]


tensor(1.6368, grad_fn=<NllLossBackward0>)


100%|██████████| 600/600 [00:04<00:00, 131.23it/s]


tensor(0.2720, grad_fn=<NllLossBackward0>)


100%|██████████| 600/600 [00:04<00:00, 144.97it/s]


tensor(0.1735, grad_fn=<NllLossBackward0>)


100%|██████████| 600/600 [00:04<00:00, 133.01it/s]


tensor(0.1339, grad_fn=<NllLossBackward0>)


100%|██████████| 600/600 [00:04<00:00, 125.72it/s]

tensor(0.1116, grad_fn=<NllLossBackward0>)


## GPU Training Loop

In [14]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
    print("running on the GPU")
else:
    device = torch.device("cpu")
    print("running on the CPU")

running on the GPU


In [15]:
net = Conv_NN(n_channels=1).to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001)

In [16]:
epochs = 10
BATCH_SIZE = 100
for epoch in range (epochs):
    for i in tqdm(range(0,len(train_images), BATCH_SIZE)):
       
        batch_data = train_images[i:i+BATCH_SIZE].to(device)
        batch_labels = train_labels[i:i+BATCH_SIZE].to(device)
        
        net.zero_grad()  
        output = net(batch_data)  
        
        loss = loss_function(output, batch_labels)  
        loss.backward()  
        optimizer.step()  
    
    print(loss)

100%|██████████| 600/600 [00:08<00:00, 69.12it/s] 


tensor(2.2941, device='cuda:0', grad_fn=<NllLossBackward0>)


100%|██████████| 600/600 [00:01<00:00, 552.49it/s]


tensor(2.2898, device='cuda:0', grad_fn=<NllLossBackward0>)


100%|██████████| 600/600 [00:01<00:00, 505.48it/s]


...

tensor(2.0885, device='cuda:0', grad_fn=<NllLossBackward0>)


100%|██████████| 600/600 [00:01<00:00, 441.33it/s]


tensor(1.8075, device='cuda:0', grad_fn=<NllLossBackward0>)


100%|██████████| 600/600 [00:01<00:00, 539.08it/s]

tensor(1.3000, device='cuda:0', grad_fn=<NllLossBackward0>)


## Testing Loop

In [17]:
with torch.no_grad():
    predicted = net.forward(test_images.to(device))
    print(predicted[0:2], predicted[0:2].shape)
    
    
    # We want to pick our the class with highest value, and thus highest probability of being that number
    predicted_classes = torch.argmax(predicted, dim = 1)
    # Our model seems pretty good!
    print(predicted_classes[0:5])
    print(test_labels[0:5])
    
    
    
# Before we calc accuracy etc, we are done using the GPU, so let's store our data back on the CPU
# If we don't do this step, we won't be able to use functions that require data to be stored on the CPU (pretty much everything.)

device = torch.device('cpu')
predicted_classes = predicted_classes.to(device)
test_labels = test_labels.to(device)



tensor([[-3.2214, -4.0188, -2.3354, -1.9373, -3.1776, -2.7329, -3.6162, -1.0178,
         -2.6736, -1.9857],
        [-3.1777, -1.8996, -2.0845, -2.6809, -3.0206, -2.5339, -1.3486, -3.8058,
         -1.7978, -3.2138]], device='cuda:0') torch.Size([2, 10])
tensor([7, 6, 1, 0, 4], device='cuda:0')
tensor([7, 2, 1, 0, 4])


In [18]:
correct = 0
total = 0
for i in range(len(predicted_classes)):

    if predicted_classes[i] == test_labels[i]:
        correct += 1
    total += 1
print("Accuracy: ", round(correct/total, 3))

Accuracy:  0.753


In [19]:
from sklearn.metrics import confusion_matrix

In [20]:
cf_mat = confusion_matrix(test_labels, predicted_classes)
cf_mat

array([[ 935,    0,   16,    2,    0,    0,   14,    1,   12,    0],
       [   0, 1070,   47,    5,    0,    0,    2,    0,    8,    3],
       [  62,   67,  808,   16,    6,    0,   43,   18,    9,    3],
       [   9,   27,   63,  794,    0,    8,   28,   21,   31,   29],
       [  32,   17,   10,    7,  705,    0,   41,    2,   62,  106],
       [  52,   32,   11,   79,   40,  332,  120,   23,  191,   12],
       [  48,   29,   12,    0,   20,    3,  810,    0,   36,    0],
       [  30,   59,   47,    5,    4,    1,    0,  844,    6,   32],
       [  40,   50,   20,   46,   18,   30,   70,   21,  638,   41],
       [  38,   23,   15,   18,  161,    3,    5,   93,   54,  599]],
      dtype=int64)

## Deep Models

Deep Models can be great! To learn


https://www.run.ai/guides/deep-learning-for-computer-vision/deep-convolutional-neural-networks


In [21]:
class ConvDeep(nn.Module):
    def __init__(self, n_channels):
        super().__init__()

        # Convolutional Layers & Max Pooling Layer Defined:
        self.conv1 = nn.Conv2d(in_channels = n_channels, out_channels = 32, kernel_size = (3,3) )
        self.conv2 = nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size = (3,3) )
        self.conv3 = nn.Conv2d(in_channels = 64, out_channels = 128, kernel_size = (3,3) )
        self.conv4 = nn.Conv2d(in_channels = 128, out_channels = 256, kernel_size = (3,3) )
        self.conv5 = nn.Conv2d(in_channels = 256, out_channels = 512, kernel_size = (3,3) )
        self.conv6 = nn.Conv2d(in_channels = 512, out_channels = 1024, kernel_size = (3,3) )

        self.pool = nn.MaxPool2d((2,2))


    def forward(self, x):
        # print(x.shape)
        x = F.relu(self.conv1(x))
        x = self.pool(x)

        # print(x.shape)

        x = F.relu(self.conv2(x))
        x = self.pool(x)

        # print(x.shape)

        x = F.relu(self.conv3(x))
        x = self.pool(x)

        # print(x.shape)

        x = F.relu(self.conv4(x))
        x = self.pool(x)

        # print(x.shape)

        x = F.relu(self.conv5(x))
        x = self.pool(x)

        # print(x.shape)

        x = F.relu(self.conv6(x))
        x = self.pool(x)

        # print(x.shape)

        print(f"Step 1: Shape Before Flattening: {x.shape}")
        

In [22]:
net = ConvDeep(n_channels=1)

In [23]:

n_samples = 10
test_output = net.forward(train_images[0:n_samples])

RuntimeError: Calculated padded input size per channel: (1 x 1). Kernel size: (3 x 3). Kernel size can't be greater than actual input size

In [28]:
class ConvWide(nn.Module):
    def __init__(self, n_channels):
        super().__init__()

        # Convolutional Layers & Max Pooling Layer Defined:
        self.conv1 = nn.Conv2d(in_channels = n_channels, out_channels = 128, kernel_size = (3,3) )
        self.conv2 = nn.Conv2d(in_channels = 128, out_channels = 512, kernel_size = (3,3))

        self.pool = nn.MaxPool2d((2,2))


        # Step 2: Flatten 
        self.flatten = nn.Flatten(1)


        self.fc1 = nn.Linear(512*5*5, 4000)
        self.fc2 = nn.Linear (4000, 10)


    def forward(self, x):
        # print(x.shape)
        x = F.relu(self.conv1(x))
        x = self.pool(x)

        # print(x.shape)

        x = F.relu(self.conv2(x))
        x = self.pool(x)


        #print(f"Step 1: Shape Before Flattening: {x.shape}")

        x = self.flatten(x)

        x = F.relu(self.fc1(x))
        x = F.log_softmax(self.fc2(x), dim = 1)

        #print(f"Step 2: Shape of Output {x.shape}")

        return x
        

In [29]:
net = ConvWide(n_channels=1)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001)

In [30]:
n_samples = 10
test_output = net.forward(train_images[0:n_samples])

In [31]:
net = net.to(device)

epochs = 10
BATCH_SIZE = 100
for epoch in range (epochs):
    for i in tqdm(range(0,len(train_images), BATCH_SIZE)):
       
        batch_data = train_images[i:i+BATCH_SIZE].to(device)
        batch_labels = train_labels[i:i+BATCH_SIZE].to(device)
        
        net.zero_grad()  
        output = net(batch_data)  
        
        loss = loss_function(output, batch_labels)  
        loss.backward()  
        optimizer.step()  
    
    print(loss)

100%|██████████| 600/600 [03:39<00:00,  2.73it/s]


tensor(2.0661, grad_fn=<NllLossBackward0>)


100%|██████████| 600/600 [03:43<00:00,  2.69it/s]


tensor(1.4688, grad_fn=<NllLossBackward0>)


100%|██████████| 600/600 [03:47<00:00,  2.64it/s]


...

tensor(0.3287, grad_fn=<NllLossBackward0>)


100%|██████████| 600/600 [03:48<00:00,  2.63it/s]


tensor(0.2999, grad_fn=<NllLossBackward0>)


100%|██████████| 600/600 [03:46<00:00,  2.65it/s]

tensor(0.2761, grad_fn=<NllLossBackward0>)


In [32]:
with torch.no_grad():
    predicted = net.forward(test_images.to(device))
    print(predicted[0:2], predicted[0:2].shape)
    
    
    # We want to pick our the class with highest value, and thus highest probability of being that number
    predicted_classes = torch.argmax(predicted, dim = 1)
    # Our model seems pretty good!
    print(predicted_classes[0:5])
    print(test_labels[0:5])
    
    
    
# Before we calc accuracy etc, we are done using the GPU, so let's store our data back on the CPU
# If we don't do this step, we won't be able to use functions that require data to be stored on the CPU (pretty much everything.)

device = torch.device('cpu')
predicted_classes = predicted_classes.to(device)
test_labels = test_labels.to(device)


tensor([[-1.3233e+01, -1.6123e+01, -1.3341e+01, -8.8517e+00, -1.3945e+01,
         -1.2711e+01, -1.8405e+01, -7.5002e-04, -1.1817e+01, -7.4323e+00],
        [-6.2752e+00, -7.2123e+00, -8.8758e-02, -5.1278e+00, -1.5350e+01,
         -4.0843e+00, -3.0914e+00, -1.6895e+01, -4.2611e+00, -1.5441e+01]]) torch.Size([2, 10])
tensor([7, 2, 1, 0, 4])
tensor([7, 2, 1, 0, 4])


In [33]:
correct = 0
total = 0
for i in range(len(predicted_classes)):

    if predicted_classes[i] == test_labels[i]:
        correct += 1
    total += 1
print("Accuracy: ", round(correct/total, 3))

Accuracy:  0.918


In [34]:
cf_mat = confusion_matrix(test_labels, predicted_classes)
cf_mat

array([[ 961,    0,    1,    2,    0,    3,    7,    1,    5,    0],
       [   0, 1110,    4,    3,    0,    2,    3,    0,   13,    0],
       [  11,    1,  916,   12,   14,    1,   19,   22,   32,    4],
       [   1,    2,   17,  913,    0,   24,    0,   18,   29,    6],
       [   1,    2,    3,    0,  898,    0,   19,    4,    5,   50],
       [  11,    4,    5,   32,    9,  788,   14,    2,   20,    7],
       [  12,    4,    9,    1,   11,   20,  897,    1,    3,    0],
       [   1,   11,   30,    3,    6,    0,    0,  924,    7,   46],
       [   8,    4,    6,   22,    9,   18,    7,   14,  865,   21],
       [  10,    5,    6,   11,   27,    7,    0,   25,   10,  908]],
      dtype=int64)

## Tutorial

## Question 1:

**Create a Convolutional Neural Network, that has the following structure:**

<u> Part 1:</u>

Each Layer discussed below should use a Kernal Size of $(3,3)$ and a Max Pooling Size of $(2,2)$

- An Input Convolutional Layer with 1 Input Channel & 16 Output Channels.
- A Second Hidden Convolutiona Layer with 32 Output Channels. 
- A Third Hidden Convolutional Layer with 64 Output Channels.


<u> Part 2: </u>

Flatten your data appropriately after completing Part 1

- A Linear Layer with output shape of 100
- A Linear Layer with output shape of 50
- A Final **Output** Layer with shape of *Number of Classes*

In [ ]:
class YourModelNameHere():
    
    def __init__(self, n_channels):
        super().__init__()

        #TODO: Student to Fill in Code Here

    def forward(self, x):
        
        #TODO: Student to Fill in Code Here.

        return x

In [ ]:
net = YourModelNameHere(n_channels = 1)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
    print("running on the GPU")
else:
    device = torch.device("cpu")
    print("running on the CPU")


net = net.to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001)

In [ ]:
epochs = STUDENT TO DECIDE
BATCH_SIZE =  STUDENT TO DECIDE
for epoch in range (epochs):
    for i in tqdm(range(0,len(train_images), BATCH_SIZE)):
       
        batch_data = train_images[i:i+BATCH_SIZE].to(device)
        batch_labels = train_labels[i:i+BATCH_SIZE].to(device)
        
        net.zero_grad()  
        output = net(batch_data)  
        
        loss = loss_function(output, batch_labels)  
        loss.backward()  
        optimizer.step()  
    
    print(loss)

In [ ]:
with torch.no_grad():
    predicted = net.forward(test_images.to(device))
    print(predicted[0:2], predicted[0:2].shape)
    
    
    # We want to pick our the class with highest value, and thus highest probability of being that number
    predicted_classes = torch.argmax(predicted, dim = 1)
    # Our model seems pretty good!
    print(predicted_classes[0:5])
    print(test_labels[0:5])
    
    
    
# Before we calc accuracy etc, we are done using the GPU, so let's store our data back on the CPU
# If we don't do this step, we won't be able to use functions that require data to be stored on the CPU (pretty much everything.)

device = torch.device('cpu')
predicted_classes = predicted_classes.to(device)
test_labels = test_labels.to(device)


In [ ]:
correct = 0
total = 0
for i in range(len(predicted_classes)):

    if predicted_classes[i] == test_labels[i]:
        correct += 1
    total += 1
print("Accuracy: ", round(correct/total, 3))

In [ ]:
from sklearn.metrics import confusion_matrix
cf_mat = confusion_matrix(test_labels, predicted_classes)
cf_mat

## Question 2:

Experiment with Creating a Neural Network with **5 Convolutional Layers & 2 Linear Layers**. Follow the same steps as above.

Use the number of output channels, linear output nodes, Kernel Size, Max Pooling Size, that you want!
